In [1]:
import os, json, pandas as pd, torch

In [2]:
os.makedirs("local_database/ToS_100", exist_ok=True)
os.makedirs("local_database/KB", exist_ok=True)
csv_path = "local_database/ToS_100/ToS_100.csv"

In [ ]:
df = pd.DataFrame({
    "document_ID": [1,1,2,2,3,3],
    "text": [
        "we may collect your data for analytics",          # A=1
        "cookies improve your experience",                 # A=1
        "we will not share data with third parties",       # A=0
        "third parties might receive aggregated stats",    # A=1
        "unsubscribe anytime via settings",                # A=0
        "personal data processed lawfully",                # A=1
    ],
    "A": [1,1,0,1,0,1]
})

df["A_targets"] = ["[0,1]","[1]","[]","[0]","[]","[2]"]
df.to_csv(csv_path, index=False)

In [ ]:
with open("local_database/KB/A.txt","w") as f:
    f.write("we collect data for analytics\n")
    f.write("we use cookies to improve experience\n")
    f.write("processing of personal data must be lawful\n")


os.makedirs("configs", exist_ok=True)
json.dump({"learning_rate":2e-3,"batch_size":4,"epochs":5}, open("configs/training_config.json","w"))
json.dump({
    "experimental_basic_memn2n_v2":{
        "embedding_dim":64, "hidden_dim":64, "dropout":0.1,
        "partial_supervision_info":{"value":{"flag": True, "coefficient":1.0, "margin":0.5}}
    }
}, open("configs/distributed_model_config.json","w"))
json.dump({"category":"A"}, open("configs/data_loader.json","w"))
json.dump({"earlystopping":{"monitor":"val_f1_score","mode":"max","patience":5,"min_delta":0.0}},
          open("configs/callbacks.json","w"))

In [5]:
splits = {
    "train":[1,2],   # docs 1 y 2
    "val":[3],       # doc 3
    "test":[3],      # en ejemplo, usamos el mismo; en real, separa
}
# materializamos CSVs como espera el trainer
def mask_ids(df, ids): return df[df["document_ID"].astype(str).isin(set(map(str, ids)))]
fold_dir = "cv_test/torch/fold_1"; os.makedirs(fold_dir, exist_ok=True)
mask_ids(df, splits["train"]).to_csv(f"{fold_dir}/train.csv", index=False)
mask_ids(df, splits["val"]).to_csv(f"{fold_dir}/val.csv", index=False)
mask_ids(df, splits["test"]).to_csv(f"{fold_dir}/test.csv", index=False)

In [6]:
from unfairness import load_hparams, load_category, train_from_csv_splits
hparams = load_hparams("configs/distributed_model_config.json","configs/training_config.json")
category = load_category("configs/data_loader.json")
ckpt_path, metrics = train_from_csv_splits(
    train_csv=f"{fold_dir}/train.csv",
    val_csv=f"{fold_dir}/val.csv",
    test_csv=f"{fold_dir}/test.csv",
    category=category,
    hparams=hparams,
    out_dir=fold_dir,
    max_len=32,
    batch_size=4,
)
print("best_ckpt:", ckpt_path)
print("metrics:", metrics)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/nvme/git/contest/hackathon2025/.venv/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
You are using a CUDA device ('NVIDIA GeForce RTX 3060') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torc

/nvme/git/contest/hackathon2025/.venv/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
/nvme/git/contest/hackathon2025/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 2. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
/nvme/git/contest/hackathon2025/.venv/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
/nvme/git/contest/hackathon2025/.venv/lib/python3.13/site-packages/pyt

Epoch 0:   0%|          | 0/1 [00:00<?, ?it/s]

/nvme/git/contest/hackathon2025/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 4. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.


Epoch 20: 100%|██████████| 1/1 [00:00<00:00, 50.87it/s, v_num=3, val_f1_score=0.000, val_loss=0.657, train_loss=0.434] 


Restoring states from the checkpoint path at /nvme/git/contest/hackathon2025/cv_test/torch/fold_1/best-v2.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /nvme/git/contest/hackathon2025/cv_test/torch/fold_1/best-v2.ckpt
/nvme/git/contest/hackathon2025/.venv/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:433: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 160.47it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
         test_f1                    0.0
        test_loss           0.6626285314559937
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
best_ckpt: /nvme/git/contest/hackathon2025/cv_test/torch/fold_1/best-v2.ckpt
metrics: {'val_f1': 0.0, 'test_f1': 0.0, 'test_accuracy': 0.0}


In [ ]:
from unfairness.token import tokenizer
from unfairness.dataset import ToS, load_kb_bank, make_dataloaders
from unfairness.utils.config_loader import load_model_and_tokenizer
import torch

tok = tokenizer()
tok.fit_on_texts(df["text"].tolist())
te_ds = ToS(mask_ids(df, splits["test"]), category, tok, max_len=32)
_, _, te_loader = make_dataloaders(te_ds, None, te_ds, batch_size=4)

kb_ids, kb_mask = load_kb_bank(category, tok, max_len=32)
lit, tok_trained = load_model_and_tokenizer(ckpt_path, kb_ids=kb_ids, kb_mask=kb_mask, map_location="cuda" if torch.cuda.is_available() else "cpu")

te_ds = ToS(mask_ids(df, splits["test"]), category, tok_trained, max_len=32)
_, _, te_loader = make_dataloaders(te_ds, None, te_ds, batch_size=4)

all_logits, all_labels = [], []
device = torch.device("cpu")
lit.to(device)
with torch.no_grad():
    for batch in te_loader:
        logits, att, _ = lit(batch["input_ids"])
        all_logits.append(logits.sigmoid().cpu())
        all_labels.append(batch["labels"].cpu())
print("test proba:", torch.cat(all_logits).numpy().round(3).tolist())
print("test gold :", torch.cat(all_labels).numpy().tolist())

test proba: [0.40700000524520874, 0.4480000138282776]
test gold : [0.0, 1.0]
